# Lesson 13: Explainable AI and Regulatory Compliance - Full Solution Notebook

Course: Machine Learning for Chemical Engineering

Focus: permutation importance, local attributions, SHAP intuition, and audit loops.

## Learning goals
1. Train a black-box model for batch release.
2. Compute permutation importance.
3. Compute exact Shapley values for a tiny 3-feature model.
4. Distinguish attribution from causality.

## Colab use
Run cells from top to bottom. The notebook is self-contained and uses synthetic data so it can run from GitHub/Colab without external files.

## Full worked solution
This version contains one possible complete implementation. Use it after attempting the exercise notebook.

In [ ]:
import itertools
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

rng = np.random.default_rng(13)
n = 220
X = pd.DataFrame({
    "temperature": rng.normal(350, 12, n),
    "spectral_flag": rng.normal(0, 1, n),
    "shift_id": rng.integers(0, 3, n),
})
y = ((X["temperature"] > 365) | (X["spectral_flag"] > 1.2)).astype(int)
y = np.where((X["shift_id"] == 2) & (rng.random(n) < 0.25), 1, y)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)
model = RandomForestClassifier(n_estimators=300, random_state=0)
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test), digits=3))

perm = permutation_importance(model, X_test, y_test, n_repeats=20, random_state=0)
for name, importance in sorted(zip(X.columns, perm.importances_mean), key=lambda t: -t[1]):
    print(f"{name:14s} importance={importance:.3f}")

print("\nAudit interpretation:")
print("If shift_id is important, it may be an operational proxy rather than a causal quality mechanism.")
print("The explanation should trigger an investigation, not close it.")

## Reflection

Use the numerical result and the plots to answer:

1. What physical assumption did the model encode?
2. Where could the model fail outside the sampled domain?
3. Which validation metric would matter most in a real engineering deployment?